In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat,
    concat_ws,
    col,
    collect_set,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
#dbutils.widgets.text('destination_schema_zone', 'None', '07-Destination Schema Zone') # change for testing
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')
#destination_schema_zone = dbutils.widgets.get('destination_schema_zone')    # change for testing

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_action_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) : # change for testing
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalActionID String, SourceSystem String, PortfolioKey String, PortfolioName String, IngestionTimestamp timestamp,"
        #     "BatchId String, LegalActionKey String, LegalActionCode String, ExposureKey String, PartyKey String,"
        #     "CDI String, AgreementKey String,CaseCode string, CaseCodePrimary string, LegalActionCreatedDate timestamp, LegalActionCreatedBy String,"
        #     "LegalActionDate timestamp, LegalActionTypeCode String,LegalActionType String, LegalActionStatusCode String,"
        #     "LegalActionStatus String, LegalActionStepKey String, LegalActionStepTypeCode String,"
        #     "LegalActionStepType String, LegalActionStepStartDate timestamp,LegalActionStepEndDate timestamp,"
        #     "LegalTypeCode string,LegalType String, CompletionResultCode String, CompletionResultDate timestamp, CompletionResult String,"
        #     "LawyerPartyKey String,LawyerPartyName string, BankAccount string, Comments String, BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalAction" # θα γεμίσει τον πίνακα που θα βάλουμε εδώ
        # change for testing

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")



        
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits", "Arotron_dec_flag"] # Legal Actions QC Decommission to Arotron - notebooks run

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        #portfolio_arotron_dec = portfolio.where(col("Arotron_dec_flag") == 1)
        #portfolio_list = [row["PortfolioName"] for row in portfolio_arotron_dec.select("PortfolioName").collect()]

         # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")
#
#
        ## ================== Delete From Delta Table ==================
        #if spark.catalog.tableExists(curated_table_path) is True:
        #    delete_from_curated_table_nj(
        #        spark,
        #        curated_table_path,
        #        run_date,  # type: ignore
        #        "arotron",
        #        portfolio_list
        #    )        
        #else:
        #    logging.info("Tables does not exist, no need to delete entries for business date!")
#

        # ================== ArmFctLegalAction ==================

        ArmFctLegalAction_extra_filters = None

        if ArmFctLegalAction_extra_filters is not None:
            ArmFctLegalAction_extra_filters.append(date_filter)
            ArmFctLegalAction_final_filters = " AND ".join(ArmFctLegalAction_extra_filters)
        else:
            ArmFctLegalAction_final_filters = date_filter

        ArmFctLegalAction_selected_fields = [
            "CustomerNo",
            "ProductID",
            "AccountNo",
            "SecurityLevelCode",
            "ActionID",
            "ProposedDate",
            "ProposedActionName",
            "ProposedActionCode",
            "ActionStatusCode",
            "ActionCategoryCode",
            #"StagingDateInserted",
            "LegalTypeCode",
            "LegalTypeName",
            "ImplementedActionCode",
            "ImplementedActionName",
            "ImplementationDate",
            "ActionAssigneeTeam",
            "ActionCreateDate",
            "ImplementationNotes",
            "ProposedByName",
            "ActionAssigneeID",
            "ActionStatusName",
            "ActionAmount",
            "CustomerID",
            "businessdate",
        ]

        ArmFctLegalAction = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotronmis_ArmFctLegalAction",
            ArmFctLegalAction_final_filters,
            ArmFctLegalAction_selected_fields,
        )

        # ================== ArmFctPhaseList ==================

        ArmFctPhaseList_extra_filters = None

        if ArmFctPhaseList_extra_filters is not None:
            ArmFctPhaseList_extra_filters.append(date_filter)
            ArmFctPhaseList_final_filters = " AND ".join(ArmFctPhaseList_extra_filters)
        else:
            ArmFctPhaseList_final_filters = date_filter

        ArmFctPhaseList_selected_fields = [
            "PhaseID",
            "ProposedActionCode",
            "ProposedDate",
            "ImplementationDate",
            "EstimatedDate",
            "ForecastDate",
            "ProposalNotes",
            "ProposedByName",
            "ActionCreateDate",
            "ProposedActionName",
            "PhaseStatusName",
            "PhaseStatusCode",
            "PhaseCategory",
            "ImplementedByName",
            "PhaseExpenses",
            "ActionID",
            "businessdate",
        ]

        ArmFctPhaseList = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotronmis_ArmFctPhaseList",
            ArmFctPhaseList_final_filters,
            ArmFctPhaseList_selected_fields,
        )

        # ================== crmGN ==================

        crmGN_extra_filters = None

        if crmGN_extra_filters is not None:
            crmGN_extra_filters.append(date_filter)
            crmGN_final_filters = " AND ".join(crmGN_extra_filters)
        else:
            crmGN_final_filters = date_filter

        crmGN_selected_fields = [
            "crmGN_RowID",
            "crmGN_Comment",
            "crmSOBJ_ID",
            "businessdate",
        ]

        crmGN = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmGN",
            crmGN_final_filters,
            crmGN_selected_fields,
        )

        
        # ================== ArmDimCustomer ==================

        ArmDimCustomer_extra_filters = None

        if ArmDimCustomer_extra_filters is not None:
            ArmDimCustomer_extra_filters.append(date_filter)
            ArmDimCustomer_final_filters = " AND ".join(ArmDimCustomer_extra_filters)
        else:
            ArmDimCustomer_final_filters = date_filter

        ArmDimCustomer_selected_fields = [
            "CustomerId",
            "CustomerBranch",
            "CustomerName",
            "businessdate",
        ]

        ArmDimCustomer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotronmis_ArmDimCustomer",
            ArmDimCustomer_final_filters,
            ArmDimCustomer_selected_fields,
        )
        
        # ================== crmLOV ==================

        crmLOV_extra_filters = None

        if crmLOV_extra_filters is not None:
            crmLOV_extra_filters.append(date_filter)
            crmLOV_final_filters = " AND ".join(crmLOV_extra_filters)
        else:
            crmLOV_final_filters = date_filter

        crmLOV_selected_fields = [
            "crmLOV_ID",
            "crmLOV_ParentID",
            "businessdate",
        ]

        crmLOV = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmLOV",
            crmLOV_final_filters,
            crmLOV_selected_fields,
        )
        
        
        

        # ================== Aliases ==================
        MIS_LegalAction = ArmFctLegalAction
        MIS_PhaseLegalAction = ArmFctPhaseList
        BranchBP = ArmDimCustomer
        LOVLegalType = crmLOV
 
       
        # ================== Primary Key fields ==================
        target_table_arotron_primary_key_fields = [
            MIS_LegalAction["ActionID"] , # LegActID
            MIS_PhaseLegalAction["PhaseID"],     # LegActStepId,
            lit("arotron"),
         ]
         
        # ================== Join conditions ==================
        # ActionsComments 
        ActionsComments = (
            crmGN
            .filter(col("crmSOBJ_ID") == "SYS_CRM_BA")
            .groupBy("crmGN_RowID")
            .agg(concat_ws(" | ", collect_set(col("crmGN_Comment").cast("string"))).alias("COM_ActionComment"))
            .withColumnRenamed("crmGN_RowID", "COM_ActionID")
        )

        join_cond_leg_phase = col("mis_leg.ActionID") == col("mis_phase.ActionID")
        join_cond_leg_comments = col("mis_leg.ActionID") == col("comments.COM_ActionID")
        join_cond_leg_customer = col("mis_leg.CustomerID") == col("cust.CustomerID")
        join_cond_branch = col("cust.CustomerBranch") == col("branch.CustomerID")
        join_cond_lov = col("lov.crmLOV_ID") == col("mis_leg.ProposedActionCode")
        lov_filter = col("lov.crmLOV_ParentID").isin("SYS_BA_LEGAL_ACTION_CODES", "SYS_BA_CATEGORY_ADMIN")

        # ================== Joins ==================
        target_table_arotron = (
                    MIS_LegalAction.alias("mis_leg")
            .join(MIS_PhaseLegalAction.alias("mis_phase"), join_cond_leg_phase, "left")
            .join(ActionsComments.alias("comments"), join_cond_leg_comments, "left")
            .join(ArmDimCustomer.alias("cust"), join_cond_leg_customer, "left")
            .join(ArmDimCustomer.alias("branch"), join_cond_branch, "left")
            .join(crmLOV.alias("lov"), join_cond_lov, "inner")
            .join(portfolio,substring(MIS_LegalAction["CustomerNo"], 1, 3) == portfolio["Prefix"],"left",) 
            .filter(lov_filter)
        )

        # ================== Addition of calculated fields ==================
        target_table_arotron = target_table_arotron.withColumn(
            "LegalActionID",
            sha2(
                concat_ws(";", *target_table_arotron_primary_key_fields),
                256,
            ),
        ).withColumn(
            "ContNum",
            when(
                MIS_LegalAction["ProductID"] == "OFF_ABG_WS_CON",
                MIS_LegalAction["AccountNo"]
            ).otherwise(lit(None))
        #).withColumn(
        #    "ExportDT",
        #    to_timestamp(MIS_LegalAction["StagingDateInserted"])
        ).withColumn(
            "Comments",
            concat_ws(
                "",
                coalesce(ActionsComments["COM_ActionComment"].cast("string"), lit("")),
                when(
                    coalesce(MIS_LegalAction["ImplementationNotes"], lit("")) != "",
                    concat(
                        lit("Implementation Notes: "),
                        MIS_LegalAction["ImplementationNotes"].cast("string")
                    )
                ).otherwise(lit(""))
            )
        ).withColumn(
            "LegalActionKey",
            concat(
                MIS_LegalAction["ActionID"].cast("string"),
                coalesce(MIS_PhaseLegalAction["PhaseID"].cast("string"), lit(""))
            )
        ).distinct()

        # ================== Selection of Final Columns ==================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.LegalActionID,
            lit("arotron").cast("String").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).alias(
                "PortfolioName"
            ),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_arotron.LegalActionKey,
            MIS_LegalAction.ActionID.cast("String").alias("LegalActionCode"),
            lit(None).cast("String").alias("ExposureKey"),
            MIS_LegalAction.CustomerNo.cast("String").alias("PartyKey"),    
            MIS_LegalAction.CustomerNo.cast("String").alias("CDI"),
            target_table_arotron.ContNum.cast("String").alias("AgreementKey"),
            lit(None).cast("String").alias("CaseCode"),
            lit("1").cast("String").alias("CaseCodePrimary"),
            MIS_LegalAction.ActionCreateDate.cast("timestamp").alias("LegalActionCreatedDate"),
            MIS_LegalAction.ProposedByName.cast("String").alias("LegalActionCreatedBy"),
            MIS_LegalAction.ProposedDate.cast("timestamp").alias("LegalActionDate"),
            MIS_LegalAction.ProposedActionCode.cast("String").alias("LegalActionTypeCode"),
            MIS_LegalAction.ProposedActionName.cast("String").alias("LegalActionType"),
            MIS_LegalAction.ActionStatusCode.cast("String").alias("LegalActionStatusCode"),
            MIS_LegalAction.ActionStatusCode.cast("String").alias("LegalActionStatus"),
            MIS_PhaseLegalAction["PhaseID"].cast("String").alias("LegalActionStepKey"),
            MIS_PhaseLegalAction.ProposedActionCode.cast("String").alias("LegalActionStepTypeCode"),
            MIS_PhaseLegalAction.ProposedActionName.cast("String").alias("LegalActionStepType"),
            MIS_PhaseLegalAction.ProposedDate.cast("timestamp").alias(
                "LegalActionStepStartDate"
            ),
            MIS_PhaseLegalAction.ImplementationDate.cast("timestamp").alias("LegalActionStepEndDate"),
            MIS_PhaseLegalAction.PhaseStatusCode.cast("String").alias("PhaseStatusCode"),
            MIS_PhaseLegalAction.PhaseStatusName.cast("String").alias("PhaseStatusName"),
            MIS_PhaseLegalAction.PhaseCategory.cast("String").alias("PhaseCategory"),
            MIS_PhaseLegalAction.EstimatedDate.cast("date").alias("PhaseEstimatedDate"),
            MIS_PhaseLegalAction.ForecastDate.cast("date").alias("PhaseForecastDate"),
            MIS_PhaseLegalAction.ImplementedByName.cast("String").alias("PhaseImplementedByName"),
            MIS_PhaseLegalAction.ProposalNotes.cast("String").alias("PhaseProposalNotes"),
            MIS_PhaseLegalAction.PhaseExpenses.cast("String").alias("PhaseExpenses"),
            MIS_LegalAction.LegalTypeCode.cast("String").alias("LegalTypeCode"),
            MIS_LegalAction.LegalTypeName.cast("String").alias("LegalType"),
            MIS_LegalAction.ImplementedActionCode.cast("String").alias("CompletionResultCode"),
            MIS_LegalAction.ImplementationDate.cast("timestamp").alias("CompletionResultDate"),
            MIS_LegalAction.ImplementedActionName.cast("String").alias("CompletionResult"),
            MIS_LegalAction.ActionAssigneeID.cast("String").alias("LawyerPartyKey"),
            MIS_LegalAction.ActionAssigneeTeam.cast("String").alias("LawyerPartyName"),
            MIS_LegalAction.ActionAmount.cast("String").alias("BankAccount"),
            target_table_arotron.Comments.cast("String").alias("Comments"),
            MIS_LegalAction["BusinessDate"].cast("date").alias("BusinessDate"),
            #coalesce(portfolio.Arotron_dec_flag, lit("0")).alias("Arotron_dec_flag"), # Legal Actions QC Decommission to Arotron - notebooks run
        )

        # Legal Actions QC Decommission to Arotron - notebooks run
        # We keep only the portfolios for which the decommission of LAs has happened
        #result_df_arotron = result_df_arotron.filter("Arotron_dec_flag == 1")

        # Legal Actions QC Decommission to Arotron - notebooks run
        # We drop the column "Arotron_dec_flag" from the final dataframe
        #result_df_arotron = result_df_arotron.drop("Arotron_dec_flag")

        # ================== Append ==================

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    # DONE till here
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                #destination_schema_zone, # change for testing
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_legal_action_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix) # change for testing
